In [ ]:
                    
# Generated by Honegumi (https://arxiv.org/abs/2502.06815)
# %pip install ax-platform==0.4.3
import numpy as np
from ax.service.ax_client import AxClient, ObjectiveProperties
from ax.modelbridge.factory import Models
from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy
from botorch.acquisition import qNoisyExpectedImprovement

# define these names as variables for reuse (e.g., branin function return)
obj1_name = "acq_time"
obj2_name = "comp_error"
gs = GenerationStrategy(GenerationStep(steps=[
        GenerationStep(
            model=Models.SOBOL,
            num_trials=6,
            max_parallelism=1
        ), 
        GenerationStep(
            model=Models.BOTORCH_MODULAR,
            num_trials=-1,  # No limitation on how many trials should be produced from this step
            max_parallelism=1,  # Parallelism limit for this step, often lower than for Sobol
            model_kwargs={"botorch_acqf_class": qNoisyExpectedImprovement}
        ),
    ]))
ax_client = AxClient()

# Define your search space and objectives
ax_client.create_experiment(
    parameters=[
        {"name": "Voltage", "type": "choice", "is_ordered": True, "values": [5.00, 10.00, 15.00, 20.00, 25.00]},
        {"name": "Current", "type": "choice", "is_ordered": True, "values": [.1, .2, .4, .8, 1.6, 3.2, 6.4, 13.0, 26.0, 51.0, 100.0, 200.0, 410.0]},
        {
            "name": "x_counts",
            "type": "range",
            "bounds": [1000000, 16000000],
        },
    ],
    objectives={
        obj1_name: ObjectiveProperties(minimize=True),
        obj2_name: ObjectiveProperties(minimize=True),
    },
)


# optimization loop
for i in range(21):

    # get_next_trial performs model fitting and acquisition function evaluation
    parameterization, trial_index = ax_client.get_next_trial()

    # extract parameters
    x1 = parameterization["x1"]
    x2 = parameterization["x2"]

    c1 = parameterization["c1"]

    # evaluate the dummy objective function
    results = branin_moo(x1, x2, c1)
    # report the results back to the algorithm
    ax_client.complete_trial(trial_index=trial_index, raw_data=results)
pareto_results = ax_client.get_pareto_optimal_parameters()

                